## This demo presents the implementation for the RSPY-323 story.

In [1]:
import requests
import os
import json
import pprint
# Init environment before running a demo notebook.
from resources.utils import *

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)
session = requests.Session()
user = os.environ["JUPYTERHUB_USER"] if cluster_mode else os.environ["RSPY_HOST_USER"]
auxip_client, cadip_client, stac_client = init_demo(owner_id = user)
if os.getenv("RSPY_LOCAL_MODE") == "1":
    href = "http://rs-server-adgs:8000"
    href_staging = "http://rs-server-staging:8000"
else:
    href = os.environ["RSPY_WEBSITE"]
    href_staging = "https://rsserverstaging.dev-rspy.esa-copernicus.eu"
    session.cookies.set ("session", os.environ["RSPY_OAUTH2_COOKIE"])

adgs_collection_id = "adgs"

Auxip service: http://rs-server-adgs:8000
CADIP service: http://rs-server-cadip:8000
Catalog service: http://rs-server-catalog:8000


In [2]:
# Create a test collection 
collection = create_test_collection()

### Check the added collection with the rs-server-catalog. This collection should be empty.

In [3]:
# Check the catalog for agrosu_my_test_collection
print(f"{stac_client.href_catalog}")
result = session.get(f"{stac_client.href_catalog}/catalog/collections/{user}:{TEST_COLLECTION}/items")
catalog_collection = result.json()
#pp.pprint(result.json())
assert catalog_collection.get("type") == "FeatureCollection"
assert catalog_collection.get("context").get("returned") == 0
print(f"No items found in the '{TEST_COLLECTION}' collection")

http://rs-server-catalog:8000
No items found in the 'my_test_collection' collection


### Creating a staging body to start the staging process

In [4]:
result = session.get(f"{href}/auxip/collections/{adgs_collection_id}/items")
items_collection = result.json()
assert items_collection.get("type") == "FeatureCollection"
assert len(items_collection.get("features")) > 0

### Start the staging process for adgs

In [5]:
TIMEOUT = 10
staging_body = {
    "version": "0.2.0",
    "id": "staging",
    "title": {
        "en": "Staging"
    },
    "description": {
        "en": "A process that takes an external STAC ItemCollection, asynchronously download its assets into the RS catalog bucket and creates the corresponding STAC items in the RS catalog."
    },
    "jobControlOptions": [
        "async-execute"
    ],
    "keywords": [
        "stac",
        "staging"
    ],
    "links": [
        {
            "type": "text/html",
            "rel": "about",
            "title": "documentation",
            "href": "https://home.rs-python.eu/rs-documentation/rs-server/docs/doc/users/functionalities/#staging",
            "hreflang": "en-US"
        }
    ],
    "inputs": {
        "collection": {
            "title": "Target collection",
            "description": "The target collection identifier in the RS catalog",
            "id": "my_test_collection",
            "schema": {
                "type": "string"
            },
            "minOccurs": 1,
            "maxOccurs": 1
        },
        "items": items_collection
    },
    "outputs": {
        "result": {
            "title": "Output STAC items",
            "id": "some_output_id",
            "description": "The staged STAC ItemCollection",
            "schema": "false",
            "minOccurs": 1,
            "maxOccurs": 1
        }
    }
}
import requests
print(f"{href_staging}")
post_response = session.post(f"{href_staging}/processes/staging/execution", 
                              json=staging_body,                              
                              timeout = TIMEOUT,)

resp = json.loads(post_response.content)
pprint.PrettyPrinter(indent=4).pprint(resp)

job_id = resp["status"]["running"]
print(f"job_id = {job_id}")

import time
timeout = 120
while timeout > 0:
    post_response = requests.get(f"{href_staging}/jobs/{job_id}",
                              **apikey_headers,
                              timeout = TIMEOUT,)
    try:
        resp = json.loads(post_response.content)
        pprint.PrettyPrinter(indent=4).pprint(resp)
        print("\n")
        if resp["status"] == "successful":
            print("Job COMPLETED")
            break
            
        if resp["status"] == "failed":
            print("Job FAILED")
            break
    except (    
            json.JSONDecodeError,
        ):        
        continue
    time.sleep(2)
    timeout -= 2

http://rs-server-staging:8000
{'status': {'running': '34dc9631-abcb-43a6-8037-f33ff56817ab'}}
job_id = 34dc9631-abcb-43a6-8037-f33ff56817ab
{   'created': '2025-02-04T16:46:06.496971',
    'finished': None,
    'identifier': '34dc9631-abcb-43a6-8037-f33ff56817ab',
    'location': None,
    'message': 'Sending tasks to the dask cluster',
    'mimetype': None,
    'process_id': 'staging',
    'progress': 0,
    'started': '2025-02-04T16:46:06.496971',
    'status': 'running',
    'type': 'process',
    'updated': '2025-02-04T16:46:06.553046'}


{   'created': '2025-02-04T16:46:06.496971',
    'finished': None,
    'identifier': '34dc9631-abcb-43a6-8037-f33ff56817ab',
    'location': None,
    'message': 'Finished',
    'mimetype': None,
    'process_id': 'staging',
    'progress': 100,
    'started': '2025-02-04T16:46:06.496971',
    'status': 'successful',
    'type': 'process',
    'updated': '2025-02-04T16:46:08.468588'}


Job COMPLETED


### Check the catalog for the present items.

In [6]:
# Check the catalog for agrosu_my_test_collection
TEST_COLLECTION: str = "my_test_collection"
result = session.get(f"{stac_client.href_catalog}/catalog/collections/{TEST_COLLECTION}/items")
catalog_collection = result.json()
assert catalog_collection.get("type") == "FeatureCollection"
assert len(catalog_collection.get("features")) > 0
for item in catalog_collection.get("features"):
    print(f"Item {item.get('id')} has {len(item.get('assets'))} assets")    

Item S1A_OPER_AUX_PREORB_OPOD_20240527T062732_V20240527T062732_20240527T062732.EOF has 1 assets
Item S1A_OPER_MPL_ORBSCT_20240514T150704_99999999T999999_0025.EOF has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20240214T110702_V20240214T071044_20240214T102814.EOF has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20240204T110702_V20240204T071044_20240204T102814.EOF has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20240129T110702_V20240129T071044_20240129T102814.EOF has 1 assets
Item S1A_OPER_MPL_ORBSCT_20240115T150704_99999999T999999_0025.EOF has 1 assets
Item S1A_OPER_AUX_OBMEMC_PDMC_20240106T000000.xml has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20231218T110702_V20231218T071044_20231218T102814.EOF has 1 assets
Item S1A_OPER_AUX_PREORB_OPOD_20231013T062732_V20231013T062732_20231013T062732.EOF has 1 assets
Item S1A_OPER_AUX_PREORB_OPOD_20231007T062732_V20231007T062732_20231007T062732.EOF has 1 assets


### Delete one item from the collection

In [7]:
item_to_delete = "S1A_OPER_MPL_ORBSCT_20200829T150704_99999999T999999_0025.EOF"
result = session.delete(f"{stac_client.href_catalog}/catalog/collections/{TEST_COLLECTION}/items/{item_to_delete}")
pp.pprint(result.json())

{'code': 'NotFoundError', 'description': ''}


### Delete the whole collection

In [8]:
result = session.delete(f"{stac_client.href_catalog}/catalog/collections/{TEST_COLLECTION}")
pp.pprint(result.json())

{'deleted collection': 'my_test_collection'}
